In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

# Load fresh original dataset
df = pd.read_csv("KaggleV2-May-2016.csv")

# Rename columns
df = df.rename(columns={
    'No-show': 'no_show',
    'Hipertension': 'hypertension',
    'Handcap': 'handicap'
})

# Convert target and gender
df['no_show'] = df['no_show'].astype(str).str.strip().map({'Yes': 1, 'No': 0})
df['Gender'] = df['Gender'].map({'F': 0, 'M': 1})

# Convert dates and create days_wait
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df['days_wait'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days

# Drop unnecessary/text/date columns
df = df.drop(
    ['PatientId', 'AppointmentID', 'Neighbourhood', 'ScheduledDay', 'AppointmentDay'],
    axis=1
)

# Train/test split
X = df.drop('no_show', axis=1)
y = df['no_show']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Models
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Evaluation
def evaluate_model(y_true, y_pred, y_prob, model_name):
    print(model_name)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("AUC-ROC:", roc_auc_score(y_true, y_prob))
    print("-" * 30)

print("Train/Test Split:", X_train.shape, X_test.shape)
print("\nClass Distribution:")
print(y.value_counts())

print("\nNo-show rate:", y.mean())
print("\nModel Results:")
evaluate_model(y_test, y_pred_lr, y_prob_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_dt, y_prob_dt, "Decision Tree")
evaluate_model(y_test, y_pred_rf, y_prob_rf, "Random Forest (Balanced)")

cv_scores = cross_val_score(rf, X, y, cv=5, scoring='roc_auc')
print("Random Forest Cross-Validation AUC:", cv_scores.mean())

Train/Test Split: (88421, 9) (22106, 9)

Class Distribution:
no_show
0    88208
1    22319
Name: count, dtype: int64

No-show rate: 0.20193255946510807

Model Results:
Logistic Regression
Accuracy: 0.7955306251696372
Precision: 0.3526315789473684
Recall: 0.015008960573476702
AUC-ROC: 0.6632095611584281
------------------------------
Decision Tree
Accuracy: 0.7671672849000272
Precision: 0.35596794601433995
Recall: 0.18906810035842295
AUC-ROC: 0.5947383892970466
------------------------------
Random Forest (Balanced)
Accuracy: 0.6969601013299557
Precision: 0.3129080863887494
Recall: 0.41868279569892475
AUC-ROC: 0.6394261449034745
------------------------------
Random Forest Cross-Validation AUC: 0.6182349816202409
